In [192]:
from dotenv import load_dotenv
load_dotenv()
import os

In [193]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [255]:
# tool creation

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """This function fetches the conversion factor between base currency and target currency."""
    url = f"https://v6.exchangerate-api.com/v6/{os.getenv("EXCHANGERATE_API_KEY")}/pair/{base_currency}/{target_currency}"
    
    response = requests.get(url)
    data = response.json()
    return data

@tool
def convert_currency(base_currency_value: float, conversion_factor: Annotated[float, InjectedToolArg]) -> float:
    """calculated the target currency value from a given base currency value."""
    return base_currency_value * conversion_factor

In [256]:
get_conversion_factor.args

{'base_currency': {'title': 'Base Currency', 'type': 'string'},
 'target_currency': {'title': 'Target Currency', 'type': 'string'}}

In [257]:
conversion_factor = get_conversion_factor.invoke({"base_currency": "USD", "target_currency": "INR"})
conversion_factor

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1766793601,
 'time_last_update_utc': 'Sat, 27 Dec 2025 00:00:01 +0000',
 'time_next_update_unix': 1766880001,
 'time_next_update_utc': 'Sun, 28 Dec 2025 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 90.0185}

In [258]:
convert_currency.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'number'},
 'conversion_factor': {'title': 'Conversion Factor', 'type': 'number'}}

In [259]:
converted_curr = convert_currency.invoke({"base_currency_value": 10, "conversion_factor": conversion_factor["conversion_rate"]})
converted_curr

900.1850000000001

Tool binding

In [260]:
llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash-lite")

In [261]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert_currency])
llm_with_tools

RunnableBinding(bound=ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, model='models/gemini-2.5-flash-lite', google_api_key=SecretStr('**********'), client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x0000026F246E6340>, default_metadata=(), model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_conversion_factor', 'description': 'This function fetches the conversion factor between base currency and target currency.', 'parameters': {'properties': {'base_currency': {'type': 'string'}, 'target_currency': {'type': 'string'}}, 'required': ['base_currency', 't

Tool Calling

In [262]:
query = "What is the conversion rate between USD and INR, based on that convert the amount of 10 USD to INR, "

In [263]:
messages = [HumanMessage(query)]

In [264]:
ai_message = llm_with_tools.invoke(messages)

In [265]:
ai_message

AIMessage(content='', additional_kwargs={'function_call': {'name': 'convert_currency', 'arguments': '{"base_currency_value": 10}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--ad9b91bf-064d-4f53-8e22-498b7eee811e-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'dad02c10-699b-43db-936a-d696c5643370', 'type': 'tool_call'}, {'name': 'convert_currency', 'args': {'base_currency_value': 10}, 'id': 'a2843b28-4340-4edd-9714-65145048f6ad', 'type': 'tool_call'}], usage_metadata={'input_tokens': 144, 'output_tokens': 46, 'total_tokens': 190, 'input_token_details': {'cache_read': 0}})

In [266]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'dad02c10-699b-43db-936a-d696c5643370',
  'type': 'tool_call'},
 {'name': 'convert_currency',
  'args': {'base_currency_value': 10},
  'id': 'a2843b28-4340-4edd-9714-65145048f6ad',
  'type': 'tool_call'}]

In [268]:
messages.append(ai_message)

In [276]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert_currency':
    # fetch the current arg
    tool_call['args']['conversion_factor'] = conversion_rate
    tool_message2 = convert_currency.invoke(tool_call)
    messages.append(tool_message2)



In [277]:
messages

[HumanMessage(content='What is the conversion rate between USD and INR, based on that convert the amount of 10 USD to INR, ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'convert_currency', 'arguments': '{"base_currency_value": 10}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--ad9b91bf-064d-4f53-8e22-498b7eee811e-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'dad02c10-699b-43db-936a-d696c5643370', 'type': 'tool_call'}, {'name': 'convert_currency', 'args': {'base_currency_value': 10, 'conversion_rate': 90.0185, 'conversion_factor': 90.0185}, 'id': 'a2843b28-4340-4edd-9714-65145048f6ad', 'type': 'tool_call'}], usage_metadata={'input_tokens': 144, 'output_tokens': 46, 'total_tokens':

In [278]:
llm_with_tools.invoke(messages).content

'The conversion rate from USD to INR is 90.0185.\n10 USD is equal to 900.185 INR.'